In [ ]:
!pip install datasets

In [ ]:
import pandas as pd
import os
import glob
from datasets import Dataset, DatasetDict
import numpy as np

In [ ]:
METADATA_PATH = "DAIC_participant_data.csv"
ALIGNED_DATA_DIR = "Aligned_Data"
OUTPUT_DIR = "daic_woz_dataset_full_exp"

SYMPTOM_COLS = [
    'PHQ8_1_NoInterest', 'PHQ8_2_Depressed', 'PHQ8_3_Sleep', 
    'PHQ8_4_Tired', 'PHQ8_5_Appetite', 'PHQ8_6_Failure', 
    'PHQ8_7_Concentration', 'PHQ8_8_Psychomotor'
]

In [ ]:
def load_and_preprocess():
    df_meta = pd.read_csv(METADATA_PATH)
    all_samples = []

    for _, row in df_meta.iterrows():
        p_id = str(int(row['Participant']))
        p_folder = os.path.join(ALIGNED_DATA_DIR, p_id)
        
        if not os.path.exists(p_folder):
            continue

        symptom_vector = row[SYMPTOM_COLS].values.astype(float).tolist()

        diagnosis = int(row['Depression_label'])
        total_phq_score = int(row['Depression_severity'])
        split = row['split']
        txt_files = glob.glob(os.path.join(p_folder, "*.txt"))
        gender = row["gender"]
        
        for file_path in txt_files:
            with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                content = f.read().strip()

                if len(content.split()) > 5:
                    all_samples.append({
                        "patient_id": p_id,
                        "file_name": os.path.basename(file_path),
                        "text": content,
                        "labels": symptom_vector,
                        "diagnosis": diagnosis,
                        "total_phq": total_phq_score,
                        "split": split,
                        "gender": gender
                    })

    df_final = pd.DataFrame(all_samples)
    train_ds = Dataset.from_pandas(df_final[df_final['split'] == 'train'])
    dev_ds = Dataset.from_pandas(df_final[df_final['split'] == 'dev'])
    test_ds = Dataset.from_pandas(df_final[df_final['split'] == 'test'])

    ds_dict = DatasetDict({
        'train': train_ds,
        'validation': dev_ds,
        'test': test_ds
    })

    ds_dict.save_to_disk(OUTPUT_DIR)
    
    return ds_dict

In [ ]:
df_meta = pd.read_csv(METADATA_PATH)

In [ ]:
print(len(df_meta[df_meta["gender"] == "male"]))
print(len(df_meta[df_meta["gender"] == "female"]))

In [ ]:
dataset = load_and_preprocess()

In [ ]:
from datasets import load_from_disk
ds = load_from_disk("daic_woz_dataset_full_exp")

In [ ]:
df_train = ds['train'].to_pandas()
df_dev = ds['validation'].to_pandas()
df_test = ds['test'].to_pandas()
df_all = pd.concat([df_train, df_dev, df_test])

patient_df = df_all.groupby('patient_id').agg({
    'text': lambda x: ' '.join(x),
    'diagnosis': 'first', 
    'total_phq': 'first', 
    'split': 'first',
    'labels': 'first', 
    "gender": "first"
}).reset_index()

for split_name in ['train', 'dev', 'test']:
    df_split = patient_df[patient_df['split'] == split_name]
    total_patients = len(df_split)
    if total_patients > 0:
        counts = df_split['diagnosis'].value_counts()
        print(f"{split_name.capitalize()} Set: {total_patients} Patients")
        print(f" healthy: {counts.get(0, 0)} ({counts.get(0, 0)/total_patients*100:.1f}%)")
        print(f" Depressed: {counts.get(1, 0)} ({counts.get(1, 0)/total_patients*100:.1f}%)")


In [ ]:
symptom_names = [
    'NoInterest', 'Depressed', 'Sleep', 'Tired', 
    'Appetite', 'Failure', 'Concentration', 'Psychomotor'
]

print(f"{'Symptom':<15} | {'Score'} | {'Train':<8} | {'Dev':<8} | {'Test':<8} | {'Total':<8}\n")

for i, symp in enumerate(symptom_names):
    for score in [0, 1, 2, 3]:
        if score == 0:
            print(f"{symp:<15} | {score:<5} | ", end="")
        else:
            print(f"{'':<15} | {score:<5} | ", end="")
            
        row_vals = []
        for split_name in ['train', 'dev', 'test', 'all']:
            if split_name == 'all':
                subset = patient_df
            else:
                subset = patient_df[patient_df['split'] == split_name]
                
            labels_matrix = np.array(subset['labels'].to_list())
            if len(labels_matrix) > 0:
                scores = labels_matrix[:, i]
                count = np.sum(scores == score)
                pct = (count / len(scores)) * 100
                row_vals.append(f"{pct:>5.1f}%")
            else:
                row_vals.append(f"{'N/A':>6}")
        print(f"{row_vals[0]:<8} | {row_vals[1]:<8} | {row_vals[2]:<8} | {row_vals[3]:<8}")

In [ ]:
total_all = len(patient_df)
healthy_all = len(patient_df[patient_df['diagnosis'] == 0])
depressed_all = len(patient_df[patient_df['diagnosis'] == 1])

for split_name in ['train', 'dev', 'test']:
    df_split = patient_df[patient_df['split'] == split_name]
    total_split = len(df_split)
    if total_split > 0:
        healthy = len(df_split[df_split['diagnosis'] == 0])
        depressed = len(df_split[df_split['diagnosis'] == 1])
        print(split_name.upper())
        print(f"Healthy: {healthy:<4} ({healthy/total_split*100:.1f}%)")
        print(f"Depressed: {depressed:<4} ({depressed/total_split*100:.1f}%)")


In [ ]:
patient_df

In [ ]:
print(len(patient_df[patient_df['gender'] == "male"]))
print(len(patient_df[patient_df['gender'] == "female"]))
print(len(patient_df[patient_df['gender'] == "unknown"]))